In [41]:
import os
import requests
import json
import re

import numpy as np
import pandas as pd
import psycopg2 as pg
from sqlalchemy import create_engine

In [42]:
import os
from dotenv import load_dotenv

load_dotenv()

# Define your PostgreSQL connection details
db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')


In [43]:
query = """
    SELECT 
        * 
    FROM 
        public.textual_data
"""

df = pd.read_sql(query, con=sqla_engine)

In [44]:
def article_analyze(title, text):
    payload = {
        "model" : "llama3",
        "messages" : [
            {"role" : "system",
             "content":f"""
                You are an expert in analyzing sentiments. Identify the key topics and sentiments of the following articles for the Bank Rakyat Indonesia or BRI for short, one of Indonesia's most reputable banks. Summarize the article and provide its sentiment.
                 """},
            {"role" : "user",
             "content": f"""
                Question: Identify the key topics of this articles related to BRI (Bank Rakyat Indonesia) 
                          and determine the sentiments of these article!

                Article Title: "{title}" 
                Article Content: "{text}"   

                Thought 1: Understand the content of the article and identify its key topics.
                Action 1: Analyze the article text to extract key topics.
                Observation 1: The key topics are...
                
                Thought 2: Analyze the tone and language of the article.
                Action 2: Determine the sentiment of the article text.
                Observation 2: The article text indicates positive/negative/neutral sentiment.

                Conclusion: Summarize the key topics and sentiment of the article USING THIS FORMAT ONLY AND NOTHING ELSE BESIDES THIS: 
                - sentiment (Very Negative, Negative, Slightly Negative, Neutral, Slightly Positive, Positive, Very Positive, and Not Relevant)
                - explanation (1 sentence, explain briefly why that sentiment score is chosen) 
                - topic (1 to 3 words, describe the key topic of the article)
                - summary (2 to 4 sentence, summarize the article content)
                
                Return your answer in JSON format like USING THIS FORMAT ONLY AND WRITE NOTHING ELSE BESIDES THIS:
                '{{"sentiment": "Neutral",
                   "explanation": "The article reports on a routine corporate event without expressing any emotional tone.",
                   "topic": "BRI Leadership Change",
                   "summary": "The article announces that PT Bank Rakyat Indonesia (Persero) Tbk. has decided to change its leadership, with Hendrikus Ivo leaving his position as an Independent Commissioner and Haryo Baskoro Wicaksono appointed as a new Independent Commissioner. The bank also changed the names of some directorates. The article does not express any emotions or opinions about these changes."}}'
                """}
        ] 
    }

    analysis_result = ""
    response = requests.post("http://localhost:11434/api/chat", json=payload)
    for message in response.iter_lines():
        jsonstr = json.loads(message)
        llm_answer = jsonstr["message"]["content"]
        analysis_result += llm_answer
    
    return analysis_result

In [45]:
results = []
for i, row in df.iterrows():
    article_id = row['article_id']
    date_published = row['article_date_published']
    article_title = row['article_title']
    article_content = row['article_content']
    analysis = article_analyze(article_title, article_content)
    results.append({"article_id": article_id, "date_published":date_published, "article_title": article_title, 
                    "article_content": article_content, "analysis": analysis})

results_df = pd.DataFrame(results)
results_df

,article_id,date_published,article_title,article_content,analysis
0,1,2024-03-01,RUPST BRI: Susunan Direksi Tetap dan Hanya Sat...,Jakarta – PT Bank Rakyat Indonesia (Persero) T...,"{""sentiment"": ""Neutral"",\n""explanation"": ""The ..."
1,2,2024-03-01,BRI tidak rombak jajaran direksi dalam RUPST 2024,Pewarta: Bayu SaputraEditor: Faisal Yunianto\t...,"Here is my answer in JSON format:\n\n{\n""senti..."
2,3,2024-03-01,"BRI (BBRI) Rombak Pengurus, Ini Susunan Komisa...",Nyaman tanpa iklan. Langganan BisnisPro Nyaman...,"{""sentiment"": ""Neutral"",\n""explanation"": ""The ..."
3,4,2024-03-02,BRI Umumkan Susunan Direksi Baru di RUPST 2024,BRI Umumkan Susunan Direksi Baru di RUPST 2024...,"Here is the analysis:\n\n{""sentiment"": ""Neutra..."
4,5,2023-03-13,"Gelar RUPST, Sunarso Kembali Ditunjuk Jadi Dir...",RUPST sepakat mengangkat kembali direksi yang ...,"Based on the article, here is my analysis:\n\n..."
...,...,...,...,...,...
73,74,2024-08-20,Rekrutmen BRILiaN Banking Officer Program (BBO...,PT Bank Rakyat Indonesia (Persero) Tbk adalah ...,"{""sentiment"": ""Neutral"",\n""explanation"": ""The ..."
74,75,2023-10-27,Proses Seleksi dan Rekrutmen pada PT. Bank Rak...,Sumber: brandingkan.com Penulis: Nazlina Dwi P...,"Here is the analysis in JSON format:\n\n{\n""se..."
75,76,2024-01-31,Dirut BRI Ungkap Rahasia di Balik Laba 2023 Te...,Nyaman tanpa iklan. Langganan BisnisPro Nyaman...,"Here is my analysis:\n\n{\n""sentiment"": ""Posit..."
76,77,2024-08-21,PT Bank Rakyat Indonesia (Persero) Tbk,PT Bank Rakyat Indonesia (Persero) Tbk – Banya...,"Here is the analysis:\n\n{""sentiment"": ""Slight..."


In [46]:
# Function to extract and parse the JSON-like string
def extract_and_parse_json(json_str):
    try:
        # Use regular expression to find the JSON object within the string
        json_match = re.search(r'\{.*\}', json_str, re.DOTALL)
        if json_match:
            json_part = json_match.group(0)
            return json.loads(json_part)
        else:
            return {"sentiment": None, "explanation": None, "topic": None, "summary": None}
    except json.JSONDecodeError:
        return {"sentiment": None, "explanation": None, "topic": None, "summary": None}

# Apply the function to parse the column
parsed_column = results_df["analysis"].apply(extract_and_parse_json)

# Convert the parsed column to a DataFrame
parsed_df = pd.json_normalize(parsed_column)

# Concatenate the new DataFrame with the original DataFrame
final_df = pd.concat([results_df, parsed_df], axis=1)
final_df = final_df.dropna(subset=['sentiment', 'explanation', 'topic', 'summary'], how='all')

In [47]:
def map_sentiment_to_score(df, sentiment_column, new_column):
    # Define a mapping dictionary for sentiment to score
    sentiment_score_map = {
        "Very Negative": -3,
        "Negative": -2,
        "Slightly Negative": -1,
        "Neutral": 0,
        "Slightly Positive": 1,
        "Positive": 2,
        "Very Positive": 3,
        "Not Relevant": 100
    }

    # Apply the mapping to create the new column
    df[new_column] = df[sentiment_column].map(sentiment_score_map)

    return df

# Assuming your DataFrame is named df and the sentiment is in column 'B'
final_df = map_sentiment_to_score(final_df, sentiment_column='sentiment', new_column='sentiment_score')

In [48]:
final_df

,article_id,date_published,article_title,article_content,analysis,sentiment,explanation,topic,summary,sentiment_score
0,1,2024-03-01,RUPST BRI: Susunan Direksi Tetap dan Hanya Sat...,Jakarta – PT Bank Rakyat Indonesia (Persero) T...,"{""sentiment"": ""Neutral"",\n""explanation"": ""The ...",Neutral,The article reports on a routine corporate eve...,BRI Leadership Change,The article announces that PT Bank Rakyat Indo...,0
1,2,2024-03-01,BRI tidak rombak jajaran direksi dalam RUPST 2024,Pewarta: Bayu SaputraEditor: Faisal Yunianto\t...,"Here is my answer in JSON format:\n\n{\n""senti...",Neutral,The article reports on a routine corporate eve...,BRI Leadership Change,The article announces that PT Bank Rakyat Indo...,0
2,3,2024-03-01,"BRI (BBRI) Rombak Pengurus, Ini Susunan Komisa...",Nyaman tanpa iklan. Langganan BisnisPro Nyaman...,"{""sentiment"": ""Neutral"",\n""explanation"": ""The ...",Neutral,The article reports on a routine corporate eve...,BRI Leadership Change,The article announces that PT Bank Rakyat Indo...,0
3,4,2024-03-02,BRI Umumkan Susunan Direksi Baru di RUPST 2024,BRI Umumkan Susunan Direksi Baru di RUPST 2024...,"Here is the analysis:\n\n{""sentiment"": ""Neutra...",Neutral,The article reports on a routine corporate eve...,BRI Leadership Change,The article announces that PT Bank Rakyat Indo...,0
4,5,2023-03-13,"Gelar RUPST, Sunarso Kembali Ditunjuk Jadi Dir...",RUPST sepakat mengangkat kembali direksi yang ...,"Based on the article, here is my analysis:\n\n...",Neutral,The article reports on a routine corporate eve...,BRI Leadership Change,The article announces that PT Bank Rakyat Indo...,0
...,...,...,...,...,...,...,...,...,...,...
73,74,2024-08-20,Rekrutmen BRILiaN Banking Officer Program (BBO...,PT Bank Rakyat Indonesia (Persero) Tbk adalah ...,"{""sentiment"": ""Neutral"",\n""explanation"": ""The ...",Neutral,The article reports on a job opening at Bank R...,BRI Job Opening,Bank Rakyat Indonesia (BRI) is recruiting for ...,0
74,75,2023-10-27,Proses Seleksi dan Rekrutmen pada PT. Bank Rak...,Sumber: brandingkan.com Penulis: Nazlina Dwi P...,"Here is the analysis in JSON format:\n\n{\n""se...",Neutral,The article discusses recruitment and selectio...,Bank Recruitment Selection,The article highlights the importance of a fai...,0
75,76,2024-01-31,Dirut BRI Ungkap Rahasia di Balik Laba 2023 Te...,Nyaman tanpa iklan. Langganan BisnisPro Nyaman...,"Here is my analysis:\n\n{\n""sentiment"": ""Posit...",Positive,The article reports on BRI's financial perform...,BRI Financial Performance,BRI's Director General Sunarso revealed the fa...,2
76,77,2024-08-21,PT Bank Rakyat Indonesia (Persero) Tbk,PT Bank Rakyat Indonesia (Persero) Tbk – Banya...,"Here is the analysis:\n\n{""sentiment"": ""Slight...",Slightly Positive,The article provides job opportunities and enc...,Job Search and Career Advice,The article advises readers not to force thems...,1


In [49]:
query = """
UPDATE 
    public.textual_data 
SET 
    sentiment_score = %(sentiment_score)s,
    sentiment_string = %(sentiment_string)s, 
    sentiment_main_topic = %(sentiment_main_topic)s,
    sentiment_explanation = %(sentiment_explanation)s,
    sentiment_summarized_articles = %(sentiment_summarized_articles)s
WHERE 
    article_id = %(article_id)s
"""

cur = pg_engine.cursor()
for _, row in final_df.iterrows():
    try:
        cur.execute(query, {
            'sentiment_score': row['sentiment_score'],
            'sentiment_string': row['sentiment'],
            'sentiment_main_topic': row['topic'],
            'sentiment_explanation': row['explanation'],
            'sentiment_summarized_articles': row['summary'],
            'article_id': row['article_id']
        })
    except Exception as e:
        print(f"Error with article_id: {row['article_id']}, article_title: {row['article_title']}")
        print(f"Error: {e}")
        pg_engine.rollback()  # Rollback the failed transaction
        continue

pg_engine.commit()